# Asistente de Normas Viales CDMX

Cuaderno de Colab para el asistente RAG: carga los documentos, crea fragmentos y embeddings, recupera contexto con fuentes y genera respuestas con Llama 3.2 Instruct.

Colab elimina archivos y variables de la máquina virtual cuando reinicia la sesión. El cuaderno guardado conserva el código; vuelve a ejecutar sus celdas en orden después de un reinicio.

## 1. Instalar la dependencia de lectura de PDF

Colab ya proporciona PyTorch, Transformers y Sentence Transformers. Este cuaderno no usa LangChain ni FAISS: la búsqueda semántica e híbrida se realiza con Sentence Transformers y NumPy.


In [ ]:
%pip install -q pypdf


## 2. Clonar el repositorio

El repositorio contiene los tres documentos PDF. Si ya se clonó en esta sesión, se reutiliza la carpeta.


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/SebastianVeraM/Asistente-de-Normas-Viales-con-LLama.git"
PROJECT_DIR = Path("/content/Asistente-de-Normas-Viales-con-LLama")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
elif not (PROJECT_DIR / ".git").exists():
    raise RuntimeError(f"La ruta existe, pero no contiene un repositorio Git: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("Proyecto:", Path.cwd())


## 3. Importar librerías y detectar GPU


In [ ]:
import re
import unicodedata
import numpy as np
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 4. Leer los documentos y crear fragmentos

Los fragmentos conservan el archivo y número de página de origen. El tamaño es de 1,000 caracteres con 150 caracteres de traslape.


In [ ]:
PDF_NAMES = [
    "Gaceta_Oficial_DF.pdf",
    "Programa_Hoy_No_Circula.pdf",
    "REGLAMENTO_DE_TRNSITO_DE_LA_CIUDAD_DE_MXICO_6.4.pdf",
]
TAMANO = 1000
TRASLAPE = 150
fragmentos_base = []

for nombre in PDF_NAMES:
    ruta = PROJECT_DIR / nombre
    if not ruta.is_file():
        raise FileNotFoundError(f"No se encontró el documento: {ruta}")

    lector = PdfReader(str(ruta))
    paginas_con_texto = 0

    for numero_pagina, pagina in enumerate(lector.pages, start=1):
        texto = pagina.extract_text() or ""
        if not texto.strip():
            continue
        paginas_con_texto += 1

        inicio = 0
        while inicio < len(texto):
            fin = min(inicio + TAMANO, len(texto))
            if fin < len(texto):
                espacio = texto.rfind(" ", inicio, fin)
                if espacio > inicio + TAMANO // 2:
                    fin = espacio

            contenido = texto[inicio:fin].strip()
            if contenido:
                fragmentos_base.append({
                    "texto": contenido,
                    "fuente": nombre,
                    "pagina": numero_pagina,
                })

            if fin >= len(texto):
                break
            inicio = max(fin - TRASLAPE, inicio + 1)

    print(f"{nombre}: {len(lector.pages)} páginas; {paginas_con_texto} con texto")

print("Total de fragmentos:", len(fragmentos_base))


## 5. Crear embeddings multilingües

Cada fragmento se convierte en un vector. Se reutiliza el mismo modelo para convertir las preguntas y compararlas con los documentos.


In [ ]:
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

modelo_embeddings = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
textos = [fragmento["texto"] for fragmento in fragmentos_base]

vectores = modelo_embeddings.encode(
    textos,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Matriz de embeddings:", vectores.shape)


## 6. Definir la búsqueda híbrida

La búsqueda combina similitud semántica con coincidencias de términos. Esto mejora la recuperación de reglas concretas que una búsqueda semántica sola podría dejar abajo en el ranking.


In [ ]:
PALABRAS_IGNORADAS = {
    "cual", "cuales", "donde", "para", "sobre", "entre",
    "como", "este", "esta", "desde", "hasta", "permita",
    "aplica", "aplican", "puede", "programa", "hoy", "circula",
    "encuentro", "encuentra", "encuentras", "estoy", "se",
}

def normalizar(texto):
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(
        caracter for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )

def buscar_fragmentos(consulta, cantidad=3, peso_lexico=0.20):
    vector_consulta = modelo_embeddings.encode(
        consulta,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    similitudes = vectores @ vector_consulta

    terminos = [
        palabra
        for palabra in re.findall(r"\w+", normalizar(consulta))
        if len(palabra) > 4 and palabra not in PALABRAS_IGNORADAS
    ]

    # Detecta nombres propios (por ejemplo, municipios) en la pregunta.
    # Se excluyen términos interrogativos y verbos frecuentes con PALABRAS_IGNORADAS.
    entidades = [
        normalizar(nombre)
        for nombre in re.findall(r"\b[A-ZÁÉÍÓÚÑ][a-záéíóúñ]{4,}\b", consulta)
        if normalizar(nombre) not in PALABRAS_IGNORADAS
    ]

    puntajes = similitudes.copy()
    for indice, fragmento in enumerate(fragmentos_base):
        texto = normalizar(fragmento["texto"])
        coincidencias = sum(
            termino in texto or termino.rstrip("s") in texto
            for termino in terminos
        )
        puntajes[indice] += peso_lexico * coincidencias

        # Una mención exacta de una entidad consultada debe elevar el fragmento.
        coincidencias_entidad = sum(
            bool(re.search(rf"\b{re.escape(entidad)}\b", texto))
            for entidad in entidades
        )
        puntajes[indice] += 0.80 * coincidencias_entidad

    indices = np.argsort(puntajes)[::-1][:cantidad]
    return [
        (fragmentos_base[indice], float(similitudes[indice]), float(puntajes[indice]))
        for indice in indices
    ]


## 7. Probar la recuperación

La prueba comprueba que el contexto incluya la regla localizada en el Reglamento y muestra las páginas recuperadas.


In [ ]:
PREGUNTA_PRUEBA = "¿Cuál es el límite de velocidad en estacionamientos y vías peatonales?"
resultados = buscar_fragmentos(PREGUNTA_PRUEBA, cantidad=3)

contexto = "\n\n".join(
    f"[Fuente: {fragmento['fuente']}, página {fragmento['pagina']}]\n{fragmento['texto']}"
    for fragmento, _, _ in resultados
)
contexto_limpio = normalizar(" ".join(contexto.split()))

print("Páginas recuperadas:", [fragmento["pagina"] for fragmento, _, _ in resultados])
print("Incluye la regla de 10 km/h:", "10 kilometros por hora" in contexto_limpio)

for fragmento, similitud, puntaje in resultados:
    print(
        f"\nPágina {fragmento['pagina']} | similitud {similitud:.3f} "
        f"| combinado {puntaje:.3f}"
    )


## 8. Cargar Llama 3.2 con el secreto de Hugging Face

Antes de ejecutar esta celda, agrega `HF_TOKEN` en el panel **Secretos** de Colab y habilita su acceso para este cuaderno. Usa un token de lectura de la misma cuenta a la que se concedió acceso al modelo. No escribas el token en el notebook ni lo subas al repositorio.

In [ ]:
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("No se encontró HF_TOKEN en Secretos de Colab")

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

print("Modelo cargado:", MODEL_ID)
print("Dispositivo:", model.device)

## 9. Generar una respuesta fundamentada en el RAG

La función recupera los fragmentos pertinentes y agrega los fragmentos vecinos de la misma página para conservar reglas que quedaron divididas por el tamaño de corte. El contexto elimina el traslape repetido y conserva fuente y página.

In [ ]:
def construir_contexto_ampliado(resultados_rag, vecinos=1):
    indices_contexto = set()

    for documento, _, _ in resultados_rag:
        indice = next(
            (i for i, base in enumerate(fragmentos_base) if base is documento),
            None,
        )
        if indice is None:
            continue

        for candidato in range(
            max(0, indice - vecinos),
            min(len(fragmentos_base), indice + vecinos + 1),
        ):
            vecino = fragmentos_base[candidato]
            if (
                vecino["fuente"] == documento["fuente"]
                and vecino["pagina"] == documento["pagina"]
            ):
                indices_contexto.add(candidato)

    paginas = {}
    for indice in sorted(indices_contexto):
        documento = fragmentos_base[indice]
        clave = (documento["fuente"], documento["pagina"])
        paginas.setdefault(clave, []).append(documento["texto"])

    partes_contexto = []
    for (fuente, pagina), textos_pagina in paginas.items():
        texto_combinado = textos_pagina[0]
        for texto_siguiente in textos_pagina[1:]:
            limite = min(300, len(texto_combinado), len(texto_siguiente))
            solapamiento = next(
                (
                    n for n in range(limite, 0, -1)
                    if texto_combinado.endswith(texto_siguiente[:n])
                ),
                0,
            )
            texto_combinado += texto_siguiente[solapamiento:]

        partes_contexto.append(
            f"[Fuente: {fuente}, página {pagina}]\n{texto_combinado}"
        )

    return "\n\n".join(partes_contexto)


def responder_con_rag(pregunta, cantidad=5, vecinos=1):
    resultados_rag = buscar_fragmentos(pregunta, cantidad=cantidad)
    if not resultados_rag:
        return "No encontré fragmentos pertinentes en los documentos.", resultados_rag

    contexto = construir_contexto_ampliado(resultados_rag, vecinos=vecinos)
    mensajes = [
        {
            "role": "system",
            "content": (
                "Eres un asistente de normas viales de la Ciudad de México. "
                "Responde en español de forma concisa y completa, usando el contexto. "
                "Puedes combinar hechos explícitos del contexto para obtener una "
                "conclusión directa, y debes explicar brevemente esa inferencia. "
                "Para el ámbito territorial de un programa, si el documento define "
                "una zona para ese programa y enumera el municipio dentro de esa zona, "
                "responde que el municipio está incluido en el ámbito descrito y cita "
                "la definición. No confundas pertenecer a la zona con que una "
                "restricción aplique hoy a un vehículo concreto ni con una contingencia. "
                "Si el contexto enumera explícitamente una categoría como exenta "
                "de las medidas por contingencia y preguntan si esa categoría puede "
                "circular durante la contingencia, responde que sí está exenta de esas "
                "medidas, con la fuente y página. No contradigas una exención expresa "
                "con una advertencia genérica. Aclara que la exención se limita a las "
                "medidas descritas y no garantiza ausencia de cualquier otra restricción "
                "o de una instrucción posterior de la autoridad. Distingue el Hoy No "
                "Circula ordinario de las medidas adicionales por contingencia. "
                "No inventes reglas, fechas ni citas. Si falta una premisa necesaria, "
                "di qué dato no puedes confirmar con los documentos disponibles. "
                "Indica siempre la fuente y página que respaldan la respuesta."
            ),
        },
        {
            "role": "user",
            "content": f"Contexto:\n{contexto}\n\nPregunta: {pregunta}",
        },
    ]

    entradas = tokenizer.apply_chat_template(
        mensajes,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        salida = model.generate(
            **entradas,
            max_new_tokens=350,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    respuesta = tokenizer.decode(
        salida[0][entradas["input_ids"].shape[-1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()
    return respuesta, resultados_rag

VECTOR_PREGUNTAS = ("¿Cuál es el límite de velocidad en estacionamientos y vías peatonales donde se permite el acceso a vehículos?",
                    "¿Aplica el programa Hoy No Circula si me encuentro en Ecatepec?",
                    "¿Afecta sólamente a vehículos privados el programa Hoy No Circula?",
                    "¿Puedo circular si mi vehículo es eléctrico cuando existe una contingencia ambiental?",
                    "¿En qué fuentes puedo consultar el programa Hoy No Circula para saber si está activado?",
                    "¿Qué sucede si entro al área metropolitana con un auto con placas foraneas en plena contingencia ambiental?",
                    "¿Cuáles son los horarios del programa Hoy No Circula?",
                    "¿Qué sucede si manejo un vehículo experimental como un go-kart o un buggie dentro de una contingencia ambiental?"
                    )

PREGUNTA = VECTOR_PREGUNTAS[4]
respuesta, fuentes_recuperadas = responder_con_rag(PREGUNTA)
print(respuesta)
print("\nFragmentos recuperados:")
for documento, similitud, puntaje in fuentes_recuperadas:
    print(f"- {documento['fuente']}, página {documento['pagina']} (similitud {similitud:.3f})")